# Patterns of Recorded Civilian Targeting in Gaza
## Phase 2: Data Preprocessing
**DS312: Data Mining and Application | Group 7 | A.Y. 2026–2027**  
**Instructor:** Nicole S. Menorias  
**Group members:** Abdula, Aque, Bautro, Budlong, Mashburn, Orano, Pandi, Prollo

Following our consultation, we narrowed the study to Gaza so that our summaries and models address one area's recorded activity. We kept the existing ACLED export (January 1, 2022 to September 30, 2025). It contains 26,664 Gaza events, including 10,714 candidate Palestinian civilian-targeting events: 63 before October 7, 2023 and 10,651 on or after that date.

**Descriptive comparison.** We kept both periods in the Gaza master table to compare counts, events per week, and the share of recorded political violence represented by focal events. The periods have different lengths. The 63 earlier events provide historical context.

**Modelling preparation.** We prepared post–October 7 focal event coordinates for DBSCAN and weekly Gaza counts for anomaly detection. October 7 defines the modelling period; it is not an input variable or the train/test cutoff. In Phase 4, earlier weeks within this period will provide the reference for evaluating later weeks. We also prepared district summaries to see whether there is enough usable history for a district-level model.

### 2.1 Raw data profile

We checked the original file against our download manifest before making a working copy. The profile records missing values, distinct values, duplicates, and numeric ranges so that each cleaning decision has evidence behind it.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
# Load the original CSV.
filename = "acled_palestine_events_2022_2025.csv"
root = Path.cwd()  # Run from the project folder.
raw_path = root / "data/raw" / filename
manifest = json.loads((root / "data/metadata/acled_events_download_manifest.json").read_text(encoding="utf-8"))
raw_hash = hashlib.sha256(raw_path.read_bytes()).hexdigest()
raw = pd.read_csv(raw_path, low_memory=False)

In [3]:
# Confirm this is the documented download.
print("File matches documented download:", raw_hash == manifest["sha256"])
print("Row count matches manifest:", len(raw) == manifest["rows"])
print("Columns match manifest:", list(raw.columns) == manifest["columns"])

File matches documented download: True
Row count matches manifest: True
Columns match manifest: True


In [4]:
# Check missing values and data types.
def profile(df):
    return pd.DataFrame({
        "Type": df.dtypes.astype(str), "Missing": df.isna().sum(),
        "Missing (%)": df.isna().mean().mul(100).round(2),
        "Distinct nonmissing": df.nunique(),
    })

In [5]:
raw_profile = profile(raw)
display(raw_profile)

,Type,Missing,Missing (%),Distinct nonmissing
event_id_cnty,str,0,0.00,58842
event_date,str,0,0.00,1369
year,int64,0,0.00,4
time_precision,int64,0,0.00,3
disorder_type,str,0,0.00,4
event_type,str,0,0.00,6
sub_event_type,str,0,0.00,21
actor1,str,0,0.00,143
assoc_actor_1,str,48398,82.25,686
inter1,str,0,0.00,8


In [6]:
# Check ranges and duplicates.
display(raw[["latitude", "longitude", "fatalities", "geo_precision", "time_precision"]].describe().round(2))
print("Date labels:", raw.event_date.min(), "to", raw.event_date.max())
print("Exact duplicates:", raw.duplicated().sum())
print("Repeated event IDs:", raw.event_id_cnty.duplicated().sum())

Date labels: 2022-01-01 to 2025-09-30
Exact duplicates: 0
Repeated event IDs: 0


,latitude,longitude,fatalities,geo_precision,time_precision
count,58842.00,58842.00,58842.00,58842.00,58842.00
mean,31.75,34.83,1.18,1.33,1.01
std,0.37,0.43,5.86,0.48,0.08
min,31.24,24.11,0.00,1.00,1.00
25%,31.45,34.41,0.00,1.00,1.00
50%,31.58,35.03,0.00,1.00,1.00
75%,32.11,35.22,0.00,2.00,1.00
max,33.56,35.55,471.00,3.00,3.00


### 2.2 Cleaning and scope

We kept records coded to Palestine and assigned to Gaza Strip. We retained events with unknown districts for Gaza-wide summaries and coordinate-based analysis. We preserved missing actor fields and original targeting labels. A blank targeting label means ACLED did not flag the event; it does not establish that civilians were unharmed. We removed the empty `admin3` field from the working copy and checked key categories for whitespace and case variants before deciding whether any needed changes.

In [7]:
scope = raw.country.eq("Palestine") & raw.admin1.eq("Gaza Strip")
print("admin3 is entirely empty:", raw.admin3.isna().all())
master = raw.loc[scope].drop(columns="admin3").copy()
print("Gaza records kept:", len(master))
print("Records outside Gaza:", (~scope).sum())
print("Missing districts kept:", master.admin2.isna().sum())

admin3 is entirely empty: True
Gaza records kept: 26664
Records outside Gaza: 32178
Missing districts kept: 4


In [8]:
# Set date and precision types.
master["event_date"] = pd.to_datetime(master.event_date, format="%Y-%m-%d")
precision = ["geo_precision", "time_precision"]
print("Valid precision codes:", master[precision].isin([1, 2, 3]).all().all())
master[precision] = master[precision].astype(pd.CategoricalDtype([1, 2, 3], ordered=True))

Valid precision codes: True


In [9]:
# Validate the cleaned event fields.
print("Event IDs present and unique:", master.event_id_cnty.notna().all() and master.event_id_cnty.is_unique)
print("Coordinates in range:", master.latitude.between(-90, 90).all() and master.longitude.between(-180, 180).all())
print("Non-negative fatality counts:", master.fatalities.ge(0).all())
print("Dates match recorded years:", master.event_date.dt.year.eq(master.year).all())

Event IDs present and unique: True
Coordinates in range: True
Non-negative fatality counts: True
Dates match recorded years: True


In [10]:
# Check for inconsistent spelling caused by spaces or capitalization.
fields = ["country", "admin1", "admin2", "disorder_type", "event_type",
          "sub_event_type", "actor1", "actor2", "civilian_targeting"]
variants = []
for field in fields:
    labels = master[field].dropna()
    if labels.ne(labels.str.strip()).any() or labels.nunique() != labels.str.casefold().nunique():
        variants.append(field)
print("Fields with whitespace or case variants:", variants or "None")

Fields with whitespace or case variants: None


### 2.3 Features and analysis tables

We added an October 7 period label for descriptive comparisons, separate from the rolling anomaly baseline. We counted a focal event only when it matched the Phase 1 targeting and actor labels. We matched political violence within the semicolon-separated disorder labels so that records with more than one label could still qualify for the comparison group. Our actor rule identifies candidate records; it does not assign responsibility. We retained the notes and sources for verification.

We flagged reported fatality totals above the raw snapshot's 99th percentile for review. The 25th and 75th percentiles are both zero, so an ordinary IQR rule would flag every positive count. We retained all valid fatality values, including the flagged records. This review threshold is descriptive and will not enter later anomaly models.

In [11]:
# Label dates for the descriptive comparison and weekly counts.
cutoff = pd.Timestamp("2023-10-07")
before_label = "Before Oct 7, 2023"
after_label = "Oct 7, 2023 onward"
post_oct7 = master.event_date >= cutoff
master["period"] = np.where(post_oct7, after_label, before_label)
master["week_start"] = master.event_date.dt.to_period("W-FRI").dt.start_time

In [12]:
# Flag recorded targeting and political violence.
master["targeting_flagged"] = master.civilian_targeting.eq("Civilian targeting").fillna(False)
master["political_violence"] = master.disorder_type.str.contains("Political violence", regex=False, na=False)

In [13]:
# Identify the focal events involving Palestinian civilians.
civilian = "Civilians (Palestine)"
actor1_is_civilian = master.actor1.eq(civilian).fillna(False)
actor2_is_civilian = master.actor2.eq(civilian).fillna(False)
master["focal_event"] = master.targeting_flagged & (actor1_is_civilian | actor2_is_civilian)
master["district_missing"] = master.admin2.isna()
print("Focal events outside the political-violence group:", (master.focal_event & ~master.political_violence).sum())

Focal events outside the political-violence group: 0


In [14]:
# Flag high fatality totals for review without removing events.
severity_cutoff = raw.fatalities.quantile(0.99)
master["high_fatalities_review"] = master.fatalities.gt(severity_cutoff)
print(f"Review threshold: more than {severity_cutoff:g} reported fatalities.")
print("Flagged in scope:", master.high_fatalities_review.sum())

Review threshold: more than 23 reported fatalities.
Flagged in scope: 576


In [15]:
# Prepare post–October 7 coordinates for clustering.
incidents = master.loc[master.focal_event & post_oct7].copy()
incidents["latitude_rad"] = np.radians(incidents.latitude)
incidents["longitude_rad"] = np.radians(incidents.longitude)
print("Focal incidents for clustering:", len(incidents))

Focal incidents for clustering: 10651


In [16]:
# Count calendar days in each period.
coverage_start = pd.Timestamp(manifest["requested_filters"]["event_date_from"])
coverage_end = pd.Timestamp(manifest["requested_filters"]["event_date_to"])
days = {
    before_label: (cutoff - coverage_start).days,
    after_label: (coverage_end - cutoff).days + 1,
}

In [17]:
# Compare counts, weekly rates, and targeting shares.
period_summary = master.groupby("period")[["political_violence", "focal_event"]].sum()
period_summary["calendar_days"] = period_summary.index.map(days)
period_summary["focal_events_per_week"] = period_summary.focal_event * 7 / period_summary.calendar_days
political_total = period_summary.political_violence.replace(0, np.nan)
period_summary["focal_share_pct"] = 100 * period_summary.focal_event / political_total
display(period_summary.round(2))

,political_violence,focal_event,calendar_days,focal_events_per_week,focal_share_pct
period,,,,,
"Before Oct 7, 2023",389,63,644,0.68,16.20
"Oct 7, 2023 onward",22364,10651,725,102.84,47.63


We built a Saturday-to-Friday calendar starting on October 7, 2023. We kept the final partial week visible and marked it as incomplete. The main temporal table covers Gaza as a whole and includes events with unknown districts. The district table omits those events only where a district is required.

The weekly counts describe what the export recorded. A week with other political-violence records can have zero focal events. When no political violence was recorded, the focal share is undefined. We kept such weeks visible but treated their focal count as unavailable for the proposed rolling reference because the export alone does not establish reporting coverage. We calculated lag features from earlier calendar weeks only. The four-week sum requires all four earlier weeks to be eligible. We did not fit DBSCAN or an anomaly detector in this phase.

In [18]:
post = master.loc[post_oct7]
known = post.loc[~post.district_missing]
calendar = pd.DataFrame({"week_start": pd.date_range(cutoff, coverage_end, freq="W-SAT")})

In [19]:
# Count the events recorded in each week.
def count_events(events, keys):
    return events.groupby(keys, as_index=False).agg(
        recorded_events=("event_id_cnty", "size"),
        political_events=("political_violence", "sum"),
        focal_events=("focal_event", "sum"),
    )

In [20]:
# Include weeks with no recorded events.
def fill_calendar(counts, areas):
    keys = areas.columns.tolist() + ["week_start"]
    weekly = areas.merge(calendar, how="cross").merge(counts, on=keys, how="left")
    columns = ["recorded_events", "political_events", "focal_events"]
    weekly[columns] = weekly[columns].fillna(0).astype(int)
    return weekly.sort_values(keys).reset_index(drop=True)

In [21]:
# Build the Gaza-wide and district tables.
gaza = master[["admin1"]].drop_duplicates()
districts = master[["admin1", "admin2"]].dropna().drop_duplicates()
gaza_counts = count_events(post, ["admin1", "week_start"])
district_counts = count_events(known, ["admin1", "admin2", "week_start"])
gaza_weekly = fill_calendar(gaza_counts, gaza)
district_weekly = fill_calendar(district_counts, districts)

In [22]:
# Mark complete weeks and calculate shares.
for weekly in [gaza_weekly, district_weekly]:
    weekly["week_end"] = weekly.week_start + pd.Timedelta(days=6)
    weekly["complete_week"] = weekly.week_end.le(coverage_end)
    weekly["political_activity_recorded"] = weekly.political_events.gt(0)
    denominator = weekly.political_events.replace(0, np.nan)
    weekly["focal_share"] = weekly.focal_events / denominator

In [23]:
# Only use complete weeks with recorded political violence for the reference.
for weekly in [gaza_weekly, district_weekly]:
    weekly["history_eligible"] = weekly.complete_week & weekly.political_activity_recorded
    weekly["focal_count_for_history"] = weekly.focal_events.where(weekly.history_eligible)

In [24]:
# Build the historical features from earlier weeks only.
def add_lags(weekly, area):
    history = weekly.groupby(area)["focal_count_for_history"]
    weekly["previous_week_focal"] = history.shift(1)
    weekly["previous_four_weeks_focal"] = history.transform(
        lambda counts: counts.shift(1).rolling(4).sum())

In [25]:
add_lags(gaza_weekly, "admin1")
add_lags(district_weekly, "admin2")
print("Gaza-wide complete weeks:", gaza_weekly.complete_week.sum())
print("Gaza-wide weeks eligible for the rolling reference:", gaza_weekly.history_eligible.sum())
display(district_weekly.groupby("admin2").history_eligible.sum().rename("Eligible weeks"))

Gaza-wide complete weeks: 103
Gaza-wide weeks eligible for the rolling reference: 103


admin2
Deir El Balah    103
Gaza City        103
Khan Yunis       103
North Gaza       101
Rafah            103
Name: Eligible weeks, dtype: int64

### 2.4 Preprocessing log

| Issue or need | Our action | Why we chose it |
|---|---|---|
| 32,151 West Bank and 27 maritime records | We excluded them from the Gaza working table. | We chose one study area. All source records remain available in the raw CSV. |
| Empty `admin3` and missing actor information | We dropped `admin3` and kept missing actor/association fields. | An empty field adds no information. Filling absent actor names would introduce unsupported identities. |
| Blank targeting labels | We kept the source field and added an explicit flag. | The flag represents ACLED's coding, not proof of harm or absence of harm. |
| Missing district labels | We marked these events and omitted them only from the district-week table. | They still contribute to Gaza-wide counts, and eligible focal events keep their coordinates for clustering. |
| Dates stored as text; precision codes stored as integers | We parsed dates and treated precision codes as ordered categories. | Dates support calendar features; precision codes support quality checks and subset comparisons. |
| No duplicate IDs or exact rows | We retained all distinct events within scope, including coincident coordinates. | Equal coordinates do not imply identical events. |
| No whitespace or case-only category variants in the checked fields | We preserved original category and actor labels. | No normalization was supported by these checks. Substantive actor groupings require a reviewed mapping. |
| Skewed reported fatalities | We added a review flag and retained the original totals. | Extreme events are relevant and a zero IQR makes a standard IQR rule unhelpful here. |
| Different units for the two methods | We derived incident coordinates in radians and a calendar-aligned weekly table. | Radians support haversine distance. The weekly measures remain interpretable in their original units. |

We did not merge independent datasets, impute records, or scale the weekly counts. No transformations were fitted on future observations.

### 2.5 Before-and-after data quality report

We show missingness for both all fields and the same retained source fields. Dropping an empty column and adding derived columns changes the overall percentage mechanically; it does not mean that previously unknown values were recovered.

In [26]:
# Compare size, duplicates, and outlier flags.
quality = pd.DataFrame()
for label, data in [("Raw", raw), ("Cleaned", master)]:
    quality.loc["Rows", label] = len(data)
    quality.loc["Columns", label] = data.shape[1]
    quality.loc["Exact duplicate rows", label] = data.duplicated().sum()
    quality.loc["Repeated event IDs", label] = data.event_id_cnty.duplicated().sum()
    quality.loc["Rows above fatality review threshold", label] = data.fatalities.gt(severity_cutoff).sum()

In [27]:
# Compare missingness across all fields and across the same source fields.
source_fields = raw.columns.drop("admin3")
for label, data in [("Raw", raw), ("Cleaned", master)]:
    quality.loc["Missing cells (%) – all fields", label] = data.isna().to_numpy().mean() * 100
    quality.loc["Missing cells (%) – retained source fields", label] = data[source_fields].isna().to_numpy().mean() * 100
display(quality.round(2))
print("Category labels corrected: 0. Types converted: event_date, geo_precision, time_precision.")

Category labels corrected: 0. Types converted: event_date, geo_precision, time_precision.


,Raw,Cleaned
Rows,58842.00,26664.00
Columns,31.00,37.00
Exact duplicate rows,0.00,0.00
Repeated event IDs,0.00,0.00
Rows above fatality review threshold,576.00,576.00
Missing cells (%) – all fields,12.11,8.23
Missing cells (%) – retained source fields,9.18,10.15


In [28]:
# Compare missing values and types by field.
display(raw_profile.add_prefix("Raw ").join(profile(master).add_prefix("Cleaned "), how="outer"))

,Raw Type,Raw Missing,Raw Missing (%),Raw Distinct nonmissing,Cleaned Type,Cleaned Missing,Cleaned Missing (%),Cleaned Distinct nonmissing
actor1,str,0.0,0.00,143.0,str,0.0,0.00,34.0
actor2,str,12359.0,21.00,78.0,str,8641.0,32.41,41.0
admin1,str,0.0,0.00,3.0,str,0.0,0.00,1.0
admin2,str,4.0,0.01,17.0,str,4.0,0.02,5.0
admin3,float64,58842.0,100.00,0.0,NaN,NaN,NaN,NaN
assoc_actor_1,str,48398.0,82.25,686.0,str,25789.0,96.72,143.0
assoc_actor_2,str,47940.0,81.47,374.0,str,21846.0,81.93,168.0
civilian_targeting,str,39545.0,67.21,1.0,str,15932.0,59.75,1.0
country,str,0.0,0.00,2.0,str,0.0,0.00,1.0
disorder_type,str,0.0,0.00,4.0,str,0.0,0.00,4.0


### 2.6 Reconciliation and export

We reconciled the four tables before exporting. The master table covers both periods for descriptive and actor questions. The focal incident and weekly tables start on October 7, 2023. The Gaza-wide table is our starting point for temporal modelling, with the district-week table available for finer analysis. CSV preserves values but not pandas category types, so later notebooks will parse dates and restore those types when needed.

In [29]:
print("Gaza event total matches:", len(master) == scope.sum())
print("Gaza weekly events match:", gaza_weekly.recorded_events.sum() == len(post))
print("Gaza weekly focal events match:", gaza_weekly.focal_events.sum() == len(incidents))
print("District weekly events match:", district_weekly.recorded_events.sum() == len(known))
print("District weekly focal events match:", district_weekly.focal_events.sum() == known.focal_event.sum())
print("Raw CSV unchanged:", hashlib.sha256(raw_path.read_bytes()).hexdigest() == raw_hash)

Gaza event total matches: True
Gaza weekly events match: True
Gaza weekly focal events match: True
District weekly events match: True
District weekly focal events match: True
Raw CSV unchanged: True


In [30]:
# Save the four analysis tables.
out_dir = root / "data/processed"
out_dir.mkdir(parents=True, exist_ok=True)
master.to_csv(out_dir / "acled_events_clean.csv", index=False)
incidents.to_csv(out_dir / "acled_focal_incidents.csv", index=False)
gaza_weekly.to_csv(out_dir / "acled_gaza_week.csv", index=False)
district_weekly.to_csv(out_dir / "acled_district_week.csv", index=False)
print("Saved the four analysis tables in data/processed.")

Saved the four analysis tables in data/processed.


### References

1. Group 7, *Phase 1: Dataset Selection and Business Problem*, revised notebook.
2. ACLED, *Palestine event-level records, January 1, 2022–September 30, 2025*, downloaded October 18, 2026. See the Phase 1 provenance and [ACLED codebook](https://acleddata.com/methodology/acled-codebook).
3. DS312 *Final Project Guide*, Section 4: Phase 2, Data Preprocessing.